# sWARm Team Wins Projections

Convert individual WAR projections + roster data into team win totals.

**Prerequisites:** Run `sWARm_future_overview.ipynb` first to generate player WAR projections.

**Inputs:**
- `predictions/future_projections_hitter_YYYY.csv`
- `predictions/future_projections_pitcher_YYYY.csv`
- `data/rosters/rosters_YYYY.csv`

**Output:** `predictions/team_wins_YYYY.csv` and `predictions/team_wins_detail_YYYY.csv`

In [ ]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute()
if project_root.name == 'notebooks':
    project_root = project_root.parent.parent
sys.path.insert(0, str(project_root))

# Import team wins pipeline
from new_pipeline.models.future_season.team_wins import (
    TeamWinsPipeline,
    generate_team_wins
)
from new_pipeline.models.future_season.team_wins.team_war_aggregator import (
    generate_team_war_breakdown
)

print("sWARm Team Wins Projection System")
print("=" * 70)
print("Roster-based team win projections using individual WAR forecasts")
print()

# Configuration
BASE_YEAR = 2025      # Year to project FROM (projections target BASE_YEAR + 1)

print(f"Configuration:")
print(f"  Base year: {BASE_YEAR}")
print(f"  Projection year: {BASE_YEAR + 1}")

## Option 1: Quick Full Pipeline (Recommended)

Run the entire pipeline in one call: load data, merge projections, allocate playing time, aggregate WAR, convert to wins, and save.

In [ ]:
# Cell 2: Generate Team Wins (Full Pipeline)

standings = generate_team_wins(base_year=BASE_YEAR, save_output=True)

print(f"\nGenerated standings for {len(standings)} teams")

## Results Overview

In [ ]:
# Cell 3: League Standings Sorted by Wins

display_cols = ['Team', 'Division', 'constrained_wins', 'projected_losses',
                'win_pct', 'total_war', 'hitter_war', 'pitcher_war']
available = [c for c in display_cols if c in standings.columns]

print("\nFull League Standings (by projected wins)")
print("=" * 70)
sorted_standings = standings[available].sort_values('constrained_wins', ascending=False)
sorted_standings.index = range(1, len(sorted_standings) + 1)
print(sorted_standings.to_string())

In [ ]:
# Cell 4: Top 5 and Bottom 5 Teams

sorted_by_wins = standings.sort_values('constrained_wins', ascending=False)

print("Top 5 Teams")
print("-" * 50)
for i, (_, row) in enumerate(sorted_by_wins.head(5).iterrows(), 1):
    print(f"  {i}. {row['Team']:<5} {int(row['constrained_wins']):>3}W - {int(row['projected_losses']):>3}L  "
          f"({row['total_war']:.1f} WAR: {row['hitter_war']:.1f}H / {row['pitcher_war']:.1f}P)")

print(f"\nBottom 5 Teams")
print("-" * 50)
for i, (_, row) in enumerate(sorted_by_wins.tail(5).iterrows(), 1):
    print(f"  {i}. {row['Team']:<5} {int(row['constrained_wins']):>3}W - {int(row['projected_losses']):>3}L  "
          f"({row['total_war']:.1f} WAR: {row['hitter_war']:.1f}H / {row['pitcher_war']:.1f}P)")

## Option 2: Step-by-Step Pipeline (For Advanced Users)

Run each stage individually for inspection and debugging.

In [ ]:
# Cell 5: Initialize Pipeline and Load Data

pipeline = TeamWinsPipeline(base_year=BASE_YEAR)
pipeline.load_data()

In [ ]:
# Cell 6: Build Team Projections (merge, allocate, aggregate, convert)

standings = pipeline.build_team_projections()

In [ ]:
# Cell 7: Inspect Enriched Roster (per-player detail)

roster = pipeline.enriched_roster
print(f"Enriched roster: {len(roster)} players")
print(f"Columns: {list(roster.columns)}")
print(f"\nProjection source breakdown:")
print(roster['projection_source'].value_counts().to_string())

# Top 15 players by adjusted WAR
print(f"\nTop 15 Players by Adjusted WAR")
print("=" * 70)
top_players = roster.nlargest(15, 'adjusted_war')[
    ['Name', 'Team', 'player_type', 'rate_war', 'playing_time_factor', 'adjusted_war']
].copy()
top_players.index = range(1, len(top_players) + 1)
print(top_players.to_string())

In [ ]:
# Cell 8: Team Drill-Down
# Change TEAM to inspect any team's roster breakdown

TEAM = 'LAD'

print(f"\n{TEAM} Roster Breakdown")
print("=" * 70)
breakdown = generate_team_war_breakdown(pipeline.enriched_roster, TEAM)
print(breakdown.to_string(index=False))

# Team totals
team_row = pipeline.team_war_df[pipeline.team_war_df['Team'] == TEAM]
if len(team_row) > 0:
    r = team_row.iloc[0]
    print(f"\n  Total WAR: {r['total_war']:.1f} (Hitters: {r['hitter_war']:.1f}, Pitchers: {r['pitcher_war']:.1f})")

In [ ]:
# Cell 9: Save Results

pipeline.save_results()

## Done!

Results saved to:
- `predictions/team_wins_YYYY.csv` - Team standings with projected wins
- `predictions/team_wins_detail_YYYY.csv` - Per-player WAR breakdown by team

To update projections, re-run `sWARm_future_overview.ipynb` first, then re-run this notebook.